<a href="https://colab.research.google.com/github/ravibalebilalu/-32_document_writer/blob/main/machine%20learning/038_kidney_desease_part_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder,MinMaxScaler
from sklearn.model_selection import train_test_split,KFold
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
from sklearn.decomposition import PCA

import keras
from keras.models import Sequential,Model
from keras.layers import Dense,Dropout
from keras.callbacks import ModelCheckpoint,EarlyStopping
from keras.optimizers import Adam


In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/Shivan118/Chronic-Kidney-Disease-Project/refs/heads/main/kidney_disease.csv")
df.head()

            age		-	age
			bp		-	blood pressure
			sg		-	specific gravity
			al		-   	albumin
			su		-	sugar
			rbc		-	red blood cells
			pc		-	pus cell
			pcc		-	pus cell clumps
			ba		-	bacteria
			bgr		-	blood glucose random
			bu		-	blood urea
			sc		-	serum creatinine
			sod		-	sodium
			pot		-	potassium
			hemo		-	hemoglobin
			pcv		-	packed cell volume
			wc		-	white blood cell count
			rc		-	red blood cell count
			htn		-	hypertension
			dm		-	diabetes mellitus
			cad		-	coronary artery disease
			appet		-	appetite
			pe		-	pedal edema
			ane		-	anemia
			class		-	class

In [ ]:
 df.shape

In [ ]:
df.columns

In [ ]:
df["classification"].value_counts()

In [ ]:
df.info()

In [ ]:
df.drop(columns="id",inplace=True)

In [ ]:
missing = pd.Series(df.isnull().sum(),name="missing_values").sort_values(ascending=False)
plt.figure(figsize=(16,9))
plt.style.use("fivethirtyeight")
sns.barplot(x=missing.index,y=missing.values)
plt.xticks(rotation=90)
plt.title("Missing Values")
plt.show()

In [ ]:
 mode = SimpleImputer(missing_values=np.nan,strategy="most_frequent")
 df_imputer = pd.DataFrame(mode.fit_transform(df))
 df_imputer.columns = df.columns


In [ ]:
 for col in df_imputer.columns:
    print(f"*******************{col} *******************")
    print(f"{df_imputer[col].unique()}")

In [ ]:
for col in df_imputer.columns:
    if df_imputer[col].dtype == "object":

        df_imputer[col] = df_imputer[col].apply(lambda x : x.replace("\t","") if isinstance(x,str) else x)
        df_imputer[col] = df_imputer[col].apply(lambda x : x.replace("?",str(df_imputer[col].mode()[0])) if isinstance(x,str) else x)




In [ ]:
df_imputer["classification"].value_counts()

In [ ]:
plt.figure(figsize=(16,9))
plt.style.use("fivethirtyeight")
sns.barplot(df_imputer["classification"])
plt.show()

In [ ]:
for col in df_imputer.columns:
    try:
        df_imputer[col] = df_imputer[col].astype(float)
    except :
        df_imputer[col] = df_imputer[col]

In [ ]:
df_imputer["dm"] = df_imputer["dm"].str.lower(). apply(lambda x: x.strip())

In [ ]:
 cat_col = df_imputer.select_dtypes("object").columns.tolist()
 num_col = df_imputer.select_dtypes(exclude="object").columns.tolist()


In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(15, 12))
axes = axes.flatten()  # Flatten 2D array to 1D for easy looping

for i, ax in enumerate(axes):
    if i < len(num_col):  # Prevent index error if num_col < 16
        sns.boxplot(data=df_imputer, x=num_col[i], ax=ax)
        ax.set_title(i)
    else:
        ax.axis('off')  # Hide extra subplot
plt.style.use("ggplot")
plt.tight_layout()
plt.show()



In [ ]:
for col in df_imputer[cat_col].columns:
    le = LabelEncoder()
    df_imputer[col] = le.fit_transform(df_imputer[col])

In [ ]:
df_imputer.to_csv("kidney_final.csv")

In [ ]:
plt.figure(figsize=(20,10))
sns.heatmap(df_imputer.corr(),annot=True,cmap="viridis")
plt.show()

In [ ]:
x = df_imputer.drop(columns="classification")
y= df_imputer["classification"]

In [ ]:
ros = RandomOverSampler()
x_os,y_os = ros.fit_resample(x,y)
sc = MinMaxScaler((-1,1))
x_os =  sc.fit_transform(x_os)
x_os.shape

In [ ]:
pca = PCA(n_components=.95)
x = pca.fit_transform(x_os)
y= y_os
x.shape

In [ ]:
x_train,x_test,y_train,y_test =  train_test_split(x,y,test_size=.2,random_state=42)


In [ ]:
def model():
    clf = Sequential()
    clf.add(Dense(15,input_shape = (x_train.shape[1],),activation="relu"))
    clf.add(Dropout(0.2))
    clf.add(Dense(15,activation="relu"))
    clf.add(Dropout(0.4))
    clf.add(Dense(1,activation="sigmoid"))
    clf.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

    return clf

model = model()

history = model.fit(x_train,y_train,validation_data = (x_test,y_test),epochs=100,verbose=1)